# Part 4 — Train B1 (λ=0) and M_human (λ>0)

Trains two models on Flickr30k using LoRA on frozen SigLIP-B/16:

| Model | Loss |
|---|---|
| **B1** | L_global only (λ=0) |
| **M_human** | L_global + λ·L_region (human Flickr30k Entities bboxes) |

Both are trained for 1 epoch to quickly check whether L_region produces a meaningful
segmentation gain over the global-only baseline.  
VOC 2012 val mIoU (200 images) is used as the quick eval metric.

**L_region** is a FILIP-style sigmoid loss:  
for each (phrase, bbox) pair in the batch, find patches inside the bbox,  
then compute max-over-patches / mean-over-tokens similarity and apply  
sigmoid contrastive loss against other items' bbox regions.

In [ ]:
# ── 0. Install dependencies ──────────────────────────────────────────────────
# Run once at the start of a Colab session.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'peft>=0.10', 'datasets', 'transformers>=4.40', 'huggingface_hub',
    'tqdm', 'Pillow', 'wandb',
], check=True)
print('Dependencies installed.')

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os, sys, io, random, getpass, zipfile, urllib.request
from pathlib import Path
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
from tqdm.auto import tqdm
import torchvision.datasets as tvd

from transformers import SiglipModel, AutoProcessor
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from peft import get_peft_model, LoraConfig
import wandb

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
CFG = dict(
    model_id       = 'google/siglip-base-patch16-384',
    eval_size      = 384,
    patch_size     = 16,

    # LoRA
    lora_rank      = 4,
    lora_alpha     = 16,
    lora_dropout   = 0.1,
    # target the attention projections in both vision and text encoders
    lora_targets   = ['q_proj', 'v_proj'],

    # Training
    batch_size     = 32,       # global loss batch
    region_batch   = 16,       # items per region loss step
    lr             = 2e-4,
    weight_decay   = 0.01,
    epochs         = 1,
    warmup_steps   = 100,
    prompt         = 'a photo of a {}',

    # Region loss
    lambda_region  = 1.0,      # weight for L_region; set 0.0 for B1 run

    # Quick eval
    eval_images    = 200,      # VOC images used for quick mIoU check
    tau_seg        = 0.0,      # from Part 2

    # Output
    ckpt_dir       = 'checkpoints',
)
CFG['n_side'] = CFG['eval_size'] // CFG['patch_size']  # 24
Path(CFG['ckpt_dir']).mkdir(exist_ok=True)
print(CFG)

In [ ]:
# ── 2b. W&B init ─────────────────────────────────────────────────────────────
# Log in once per Colab session (prompts for API key if not set).
wandb.login()

run = wandb.init(
    project = 'region-grounded',
    name    = f'b1-vs-mhuman_lam{CFG["lambda_region"]}_r{CFG["lora_rank"]}',
    config  = CFG,
    tags    = ['siglip', 'flickr30k', 'lora'],
)
print(f'W&B run: {run.url}')


In [ ]:
# ── 3. Download Flickr30k (parquet) + Entities annotations ───────────────────

# ── 3a. Flickr30k parquet (images + captions + split) ────────────────────────
REPO_ID = 'nlphuji/flickr30k'
api     = HfApi(token=os.environ['HF_TOKEN'])
pq_names = sorted(
    f for f in api.list_repo_files(REPO_ID, repo_type='dataset',
                                   revision='refs/convert/parquet')
    if f.endswith('.parquet')
)
local_pq = [
    hf_hub_download(repo_id=REPO_ID, filename=f, repo_type='dataset',
                    revision='refs/convert/parquet',
                    token=os.environ['HF_TOKEN'])
    for f in pq_names
]
hf_data = load_dataset('parquet', data_files={'all': local_pq})['all']
print(f'Flickr30k rows: {len(hf_data):,}')
print('Columns:', hf_data.column_names)

# ── 3b. Flickr30k Entities XML + sentence annotations (~25 MB) ───────────────
ENTITIES_DIR = Path('data/flickr30k_entities')
ENTITIES_DIR.mkdir(parents=True, exist_ok=True)

UTILS_URL  = 'https://raw.githubusercontent.com/bryanplummer/flickr30k_entities/master/flickr30k_entities_utils.py'
UTILS_PATH = ENTITIES_DIR / 'flickr30k_entities_utils.py'
if not UTILS_PATH.exists():
    urllib.request.urlretrieve(UTILS_URL, UTILS_PATH)
    print('Downloaded entities utils.')

ZIP_URL  = 'https://github.com/bryanplummer/flickr30k_entities/raw/master/annotations.zip'
ZIP_PATH = ENTITIES_DIR / 'annotations.zip'
ANN_DIR  = ENTITIES_DIR / 'Annotations'
SENT_DIR = ENTITIES_DIR / 'Sentences'

if not ANN_DIR.is_dir():
    if not ZIP_PATH.exists():
        print('Downloading annotations.zip (~25 MB)…')
        urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(ENTITIES_DIR)
    print('Extracted.')

if str(ENTITIES_DIR) not in sys.path:
    sys.path.insert(0, str(ENTITIES_DIR))
from flickr30k_entities_utils import get_sentence_data, get_annotations

print(f'Annotations: {len(list(ANN_DIR.iterdir())):,} XML files')
print(f'Sentences  : {len(list(SENT_DIR.iterdir())):,} txt files')

In [ ]:
# ── 4. Dataset ────────────────────────────────────────────────────────────────
# One item per image. Returns:
#   pil_image  : PIL image (original size, unprocessed)
#   caption    : one randomly sampled caption string
#   orig_size  : (W, H) of the original image — needed for bbox scaling
#   phrase_boxes: list of (phrase_str, [x1,y1,x2,y2]) in original pixel coords

class FlickrSigLIPDataset(Dataset):

    def __init__(self, hf_data, ann_dir: Path, sent_dir: Path, split: str = 'train'):
        self.ann_dir  = ann_dir
        self.sent_dir = sent_dir
        self.rows     = [r for r in hf_data if r['split'] == split]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        # decode PIL image
        img_field = row['image']
        if isinstance(img_field, PILImage.Image):
            pil = img_field.convert('RGB')
        else:
            pil = PILImage.open(io.BytesIO(img_field['bytes'])).convert('RGB')
        orig_size = pil.size  # (W, H)

        caption = random.choice(row['caption'])

        # phrase-bbox pairs from Entities annotations
        stem = row['filename'].replace('.jpg', '')
        try:
            anns  = get_annotations(str(self.ann_dir  / f'{stem}.xml'))
            sents = get_sentence_data(str(self.sent_dir / f'{stem}.txt'))
        except Exception:
            return pil, caption, orig_size, []

        phrase_boxes = []
        seen = set()
        for sent in sents:
            for phrase in sent['phrases']:
                pid = phrase['phrase_id']
                if pid in seen or pid not in anns['boxes']:
                    continue
                seen.add(pid)
                for box in anns['boxes'][pid]:
                    phrase_boxes.append((phrase['phrase'], box))

        return pil, caption, orig_size, phrase_boxes

    @staticmethod
    def collate_fn(batch):
        pils, captions, orig_sizes, phrase_boxes = zip(*batch)
        return list(pils), list(captions), list(orig_sizes), list(phrase_boxes)


train_ds = FlickrSigLIPDataset(hf_data, ANN_DIR, SENT_DIR, split='train')
val_ds   = FlickrSigLIPDataset(hf_data, ANN_DIR, SENT_DIR, split='val')
print(f'Train: {len(train_ds):,} images')
print(f'Val  : {len(val_ds):,} images')

# Sanity check one sample
pil0, cap0, sz0, pb0 = train_ds[0]
print(f'\nSample: size={sz0}  caption="{cap0[:60]}…"')
print(f'phrase-box pairs: {len(pb0)}  e.g. {pb0[0] if pb0 else "none"}')

In [ ]:
# ── 5. Load SigLIP + apply LoRA ───────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(CFG['model_id'],
                                          token=os.environ['HF_TOKEN'])
base_model = SiglipModel.from_pretrained(CFG['model_id'],
                                         token=os.environ['HF_TOKEN'])

lora_cfg = LoraConfig(
    r                = CFG['lora_rank'],
    lora_alpha       = CFG['lora_alpha'],
    lora_dropout     = CFG['lora_dropout'],
    target_modules   = CFG['lora_targets'],
    bias             = 'none',
)
model = get_peft_model(base_model, lora_cfg)
model = model.to(DEVICE)
model.print_trainable_parameters()

# logit_scale and logit_bias are SigLIP's contrastive temperature parameters.
# Keep them trainable — they are NOT in LoRA modules.
for n, p in model.named_parameters():
    if 'logit_scale' in n or 'logit_bias' in n:
        p.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)')

In [ ]:
# ── 6. Loss functions ─────────────────────────────────────────────────────────

def siglip_global_loss(img_feats, txt_feats, logit_scale, logit_bias):
    """
    SigLIP sigmoid contrastive loss for a batch of (image, text) pairs.
    img_feats: (B, D) L2-normalised image embeddings
    txt_feats: (B, D) L2-normalised text embeddings
    """
    logits = logit_scale.exp() * (img_feats @ txt_feats.T) + logit_bias
    B = logits.shape[0]
    # +1 on diagonal (matched pairs), -1 everywhere else
    labels = 2 * torch.eye(B, device=logits.device) - 1
    return -F.logsigmoid(labels * logits).mean()


def bbox_patch_mask(bbox, orig_size, n_side, eval_size):
    """
    Return a boolean mask of shape (n_side*n_side,) where True = patch center
    falls inside the bbox.

    bbox      : [x1, y1, x2, y2] in original image pixel coords
    orig_size : (W, H) of the original image
    n_side    : number of patches per side (24 for SigLIP-B/16 at 384px)
    eval_size : model input resolution (384)
    """
    W, H = orig_size
    x1, y1, x2, y2 = bbox
    # Scale bbox to eval_size coords
    sx, sy = eval_size / W, eval_size / H
    x1, y1, x2, y2 = x1*sx, y1*sy, x2*sx, y2*sy

    patch_size = eval_size / n_side
    rows = torch.arange(n_side, dtype=torch.float32)
    cols = torch.arange(n_side, dtype=torch.float32)
    # Centre coords of each patch in eval_size pixels
    cy = (rows + 0.5) * patch_size   # (n_side,)
    cx = (cols + 0.5) * patch_size   # (n_side,)
    grid_cy, grid_cx = torch.meshgrid(cy, cx, indexing='ij')  # (n_side, n_side)
    mask = (grid_cx >= x1) & (grid_cx <= x2) & (grid_cy >= y1) & (grid_cy <= y2)
    return mask.reshape(-1)   # (N,)


def filip_region_loss(patch_feats_n, phrase_token_feats, bbox_masks,
                      logit_scale, logit_bias):
    """
    FILIP-style region-phrase contrastive loss.

    patch_feats_n    : (B, N, D) L2-normalised patch features
    phrase_token_feats: list of B tensors, each (T_i, D) L2-normalised token feats
    bbox_masks       : (B, N) bool — which patches are in each item's bbox
    logit_scale, logit_bias: SigLIP temperature parameters

    For pair (i, j):
      score = mean_t [ max_{p in bbox_j} sim(patch_p, token_t) ]
    Diagonal = positive, off-diagonal = negative.
    """
    B = patch_feats_n.shape[0]
    scores = torch.zeros(B, B, device=patch_feats_n.device)

    for i in range(B):   # phrase side
        tokens = phrase_token_feats[i]   # (T, D)
        for j in range(B):   # region side
            pos = patch_feats_n[j][bbox_masks[j]]   # (K, D)
            if pos.shape[0] == 0:
                continue
            # (T, K) → max over patches → (T,) → mean
            scores[i, j] = (tokens @ pos.T).max(dim=-1).values.mean()

    logits = logit_scale.exp() * scores + logit_bias
    labels = 2 * torch.eye(B, device=logits.device) - 1
    return -F.logsigmoid(labels * logits).mean()


print('Loss functions defined: siglip_global_loss, filip_region_loss')

In [ ]:
# ── 7. Feature extraction helpers ────────────────────────────────────────────
import types

def get_image_feats(model, pixel_values):
    """Global image embedding: mean of raw encoder patches, L2-normalised.
    Bypasses untrained MHAP head (transformers 5.x issue from Part 2).
    """
    captured = []
    hook = model.vision_model.encoder.register_forward_hook(
        lambda m, inp, out: captured.append(out.last_hidden_state)
    )
    model.vision_model(pixel_values=pixel_values)
    hook.remove()
    return F.normalize(captured[0].mean(dim=1), dim=-1)   # (B, D)


def get_patch_feats(model, pixel_values):
    """Per-patch raw encoder features, L2-normalised. Shape (B, N, D)."""
    captured = []
    hook = model.vision_model.encoder.register_forward_hook(
        lambda m, inp, out: captured.append(out.last_hidden_state)
    )
    model.vision_model(pixel_values=pixel_values)
    hook.remove()
    return F.normalize(captured[0], dim=-1)   # (B, N, D)


def get_text_feats(model, input_ids, attention_mask=None):
    """Global text embedding: EOS token, L2-normalised. Shape (B, D)."""
    out = model.text_model(input_ids=input_ids, attention_mask=attention_mask)
    return F.normalize(out.last_hidden_state[:, -1, :], dim=-1)   # (B, D)


def get_phrase_token_feats(model, processor, phrases, device):
    """Per-token text features for a list of phrases. Returns list of (T_i, D) tensors."""
    inputs = processor(text=phrases, return_tensors='pt',
                       padding=True, truncation=True).to(device)
    out    = model.text_model(**inputs)
    hs     = out.last_hidden_state   # (B, T, D)
    # Return each as a separate (T, D) tensor — drop padding tokens
    result = []
    for i, phrase in enumerate(phrases):
        # Find actual token length (non-pad tokens)
        if 'attention_mask' in inputs:
            length = inputs['attention_mask'][i].sum().item()
        else:
            length = hs.shape[1]
        tok_feats = F.normalize(hs[i, :length, :], dim=-1)  # (T, D)
        result.append(tok_feats)
    return result


print('Feature extraction helpers defined.')

In [ ]:
# ── 8. Quick VOC eval ─────────────────────────────────────────────────────────
# Reuses the same eval pipeline as 02_b0_benchmark (MaskCLIP-style extraction).

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

VOC_ROOT = Path('/tmp/voc')
VOC_ROOT.mkdir(exist_ok=True)
voc_val  = tvd.VOCSegmentation(root=str(VOC_ROOT), year='2012',
                                image_set='val', download=True)
print(f'VOC 2012 val: {len(voc_val)} images')


def _maskclip_attn_fwd(self, hidden_states, attention_mask=None, **kwargs):
    return self.out_proj(self.v_proj(hidden_states)), None


def quick_voc_eval(model, processor, n_images=200, tau_seg=0.0, device=DEVICE):
    """Run MaskCLIP-style zero-shot segmentation on n_images VOC val images."""
    model.eval()
    n_fg   = len(VOC_CLASSES)
    n_cls  = n_fg + 1

    # Encode text classes once
    prompts = [f'a photo of a {c}' for c in VOC_CLASSES]
    txt_in  = processor(text=prompts, return_tensors='pt',
                        padding=True).to(device)
    with torch.no_grad():
        txt_feats = F.normalize(
            model.text_model(**txt_in).last_hidden_state[:, -1, :], dim=-1
        )   # (20, D)

    # Patch extraction with MaskCLIP-style last attention bypass
    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)

    tp = torch.zeros(n_cls, dtype=torch.float64)
    fp = torch.zeros(n_cls, dtype=torch.float64)
    fn = torch.zeros(n_cls, dtype=torch.float64)
    gt_present = torch.zeros(n_cls, dtype=torch.bool)

    try:
        for i in tqdm(range(n_images), desc='VOC eval', leave=False):
            pil_img, target = voc_val[i]
            orig_w, orig_h  = pil_img.size
            gt = torch.from_numpy(np.array(target)).long()

            pix = processor(images=pil_img, return_tensors='pt').pixel_values.to(device)
            with torch.no_grad():
                captured = []
                hook = model.vision_model.encoder.register_forward_hook(
                    lambda m, inp, out: captured.append(out.last_hidden_state.detach())
                )
                model.vision_model(pixel_values=pix)
                hook.remove()

            patch_n = F.normalize(captured[0][0], dim=-1)   # (N, D)
            sim     = patch_n @ txt_feats.T                  # (N, 20)
            n_side  = int(sim.shape[0] ** 0.5)
            sim_up  = F.interpolate(
                sim.reshape(n_side, n_side, n_fg).permute(2,0,1).unsqueeze(0).float(),
                size=(orig_h, orig_w), mode='bilinear', align_corners=False
            ).squeeze(0).permute(1, 2, 0)

            max_sim, pred = sim_up.max(dim=-1)
            pred = pred + 1
            pred[max_sim < tau_seg] = 0
            pred = pred.cpu()

            valid = (gt != 255)
            pv, gv = pred[valid], gt[valid]
            for c in range(n_cls):
                pc = (pv == c); gc = (gv == c)
                tp[c] += (pc & gc).sum()
                fp[c] += (pc & ~gc).sum()
                fn[c] += (~pc & gc).sum()
                if gc.any(): gt_present[c] = True
    finally:
        last_attn.forward = orig_fwd

    iou   = tp / (tp + fp + fn).clamp(min=1e-6)
    miou  = iou[gt_present].mean().item() * 100
    return miou


print('quick_voc_eval defined.')

In [ ]:
# ── 9. Training loop ─────────────────────────────────────────────────────────

def train_one_epoch(model, processor, train_ds, lambda_region,
                    optimizer, scheduler, device, cfg,
                    run_tag='', step_offset=0):
    model.train()
    loader = DataLoader(
        train_ds,
        batch_size   = cfg['batch_size'],
        shuffle      = True,
        num_workers  = 2,
        collate_fn   = FlickrSigLIPDataset.collate_fn,
        pin_memory   = device == 'cuda',
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

    total_loss = total_global = total_region = 0.0
    n_steps = 0

    pbar = tqdm(loader, desc=f'train {run_tag}')
    for pils, captions, orig_sizes, phrase_boxes_batch in pbar:

        # ── L_global: image ↔ caption ────────────────────────────────────────
        img_inputs = processor(images=pils, return_tensors='pt',
                               padding=True).to(device)
        txt_inputs = processor(text=captions, return_tensors='pt',
                               padding=True, truncation=True).to(device)

        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            img_feats = get_image_feats(model, img_inputs['pixel_values'])
            txt_feats = get_text_feats(model, txt_inputs['input_ids'])
            l_global  = siglip_global_loss(
                img_feats, txt_feats,
                model.logit_scale, model.logit_bias
            )

        # ── L_region: phrase ↔ bbox patches ──────────────────────────────────
        l_region = torch.tensor(0.0, device=device)

        if lambda_region > 0:
            triples = []
            for img_idx, pb_list in enumerate(phrase_boxes_batch):
                for phrase, bbox in pb_list:
                    triples.append((img_idx, phrase, bbox))

            if len(triples) >= 2:
                random.shuffle(triples)
                triples = triples[:cfg['region_batch']]

                reg_img_idxs = [t[0] for t in triples]
                reg_phrases  = [t[1] for t in triples]
                reg_bboxes   = [t[2] for t in triples]
                reg_orig_sz  = [orig_sizes[i] for i in reg_img_idxs]
                reg_pils     = [pils[i] for i in reg_img_idxs]
                reg_pix      = processor(images=reg_pils, return_tensors='pt',
                                         padding=True).pixel_values.to(device)

                n_side    = cfg['n_side']
                eval_size = cfg['eval_size']

                with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                    patch_feats_n    = get_patch_feats(model, reg_pix)
                    phrase_tok_feats = get_phrase_token_feats(
                        model, processor, reg_phrases, device
                    )
                    bbox_masks = torch.stack([
                        bbox_patch_mask(bbox, sz, n_side, eval_size)
                        for bbox, sz in zip(reg_bboxes, reg_orig_sz)
                    ]).to(device)
                    l_region = filip_region_loss(
                        patch_feats_n, phrase_tok_feats, bbox_masks,
                        model.logit_scale, model.logit_bias
                    )

        loss = l_global + lambda_region * l_region

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss   += loss.item()
        total_global += l_global.item()
        total_region += l_region.item() if isinstance(l_region, torch.Tensor) else 0.0
        n_steps      += 1

        cur_lr = scheduler.get_last_lr()[0]
        wandb.log({
            f'{run_tag}/loss':         loss.item(),
            f'{run_tag}/loss_global':  l_global.item(),
            f'{run_tag}/loss_region':  l_region.item() if isinstance(l_region, torch.Tensor) else 0.0,
            f'{run_tag}/lr':           cur_lr,
            f'{run_tag}/logit_scale':  model.logit_scale.item(),
            f'{run_tag}/logit_bias':   model.logit_bias.item(),
        }, step=step_offset + n_steps)

        pbar.set_postfix({
            'loss':   f'{total_loss/n_steps:.3f}',
            'global': f'{total_global/n_steps:.3f}',
            'region': f'{total_region/n_steps:.3f}',
        })

    stats = {
        'loss':   total_loss   / n_steps,
        'global': total_global / n_steps,
        'region': total_region / n_steps,
    }
    return stats, n_steps


def make_optimizer_scheduler(model, n_steps, cfg):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg['lr'], weight_decay=cfg['weight_decay']
    )
    def lr_lambda(step):
        if step < cfg['warmup_steps']:
            return step / max(1, cfg['warmup_steps'])
        progress = (step - cfg['warmup_steps']) / max(1, n_steps - cfg['warmup_steps'])
        return max(0.0, 0.5 * (1 + np.cos(np.pi * progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


print('Training loop defined.')


In [ ]:
# ── 10. B0 eval (frozen baseline) ────────────────────────────────────────────
b0_miou = quick_voc_eval(model, processor, n_images=CFG['eval_images'],
                          tau_seg=CFG['tau_seg'])
print(f'B0 (frozen, before training): {b0_miou:.2f}% mIoU')
wandb.log({'eval/b0_miou': b0_miou}, step=0)


In [ ]:
# ── 11. Train B1 (λ=0, global loss only) ─────────────────────────────────────
base_model_b1 = SiglipModel.from_pretrained(CFG['model_id'],
                                             token=os.environ['HF_TOKEN'])
model_b1 = get_peft_model(base_model_b1, lora_cfg).to(DEVICE)
for n, p in model_b1.named_parameters():
    if 'logit_scale' in n or 'logit_bias' in n:
        p.requires_grad_(True)

n_steps_b1     = len(train_ds) // CFG['batch_size'] * CFG['epochs']
opt_b1, sch_b1 = make_optimizer_scheduler(model_b1, n_steps_b1, CFG)

print('=== Training B1 (λ_region=0) ===')
stats_b1, b1_steps = train_one_epoch(
    model_b1, processor, train_ds,
    lambda_region=0.0,
    optimizer=opt_b1, scheduler=sch_b1,
    device=DEVICE, cfg=CFG,
    run_tag='b1', step_offset=0,
)
print(f'Train stats: {stats_b1}')

model_b1.save_pretrained(f'{CFG["ckpt_dir"]}/b1_epoch1')
print('Checkpoint saved.')

b1_miou = quick_voc_eval(model_b1, processor, n_images=CFG['eval_images'],
                          tau_seg=CFG['tau_seg'])
print(f'\nB1 VOC mIoU ({CFG["eval_images"]} images): {b1_miou:.2f}%')
wandb.log({'eval/b1_miou': b1_miou,
           'b1/epoch_loss': stats_b1['loss'],
           'b1/epoch_loss_global': stats_b1['global']}, step=b1_steps)


In [ ]:
# ── 12. Train M_human (λ=1.0, global + region loss) ──────────────────────────
base_model_mh = SiglipModel.from_pretrained(CFG['model_id'],
                                             token=os.environ['HF_TOKEN'])
model_mh = get_peft_model(base_model_mh, lora_cfg).to(DEVICE)
for n, p in model_mh.named_parameters():
    if 'logit_scale' in n or 'logit_bias' in n:
        p.requires_grad_(True)

n_steps_mh      = len(train_ds) // CFG['batch_size'] * CFG['epochs']
opt_mh, sch_mh  = make_optimizer_scheduler(model_mh, n_steps_mh, CFG)

print(f'=== Training M_human (λ_region={CFG["lambda_region"]}) ===')
stats_mh, mh_steps = train_one_epoch(
    model_mh, processor, train_ds,
    lambda_region=CFG['lambda_region'],
    optimizer=opt_mh, scheduler=sch_mh,
    device=DEVICE, cfg=CFG,
    run_tag='mhuman', step_offset=b1_steps,
)
print(f'Train stats: {stats_mh}')

model_mh.save_pretrained(f'{CFG["ckpt_dir"]}/mhuman_lam{CFG["lambda_region"]}_epoch1')
print('Checkpoint saved.')

mh_miou = quick_voc_eval(model_mh, processor, n_images=CFG['eval_images'],
                          tau_seg=CFG['tau_seg'])
print(f'\nM_human VOC mIoU ({CFG["eval_images"]} images): {mh_miou:.2f}%')
wandb.log({'eval/mhuman_miou': mh_miou,
           'mhuman/epoch_loss': stats_mh['loss'],
           'mhuman/epoch_loss_global': stats_mh['global'],
           'mhuman/epoch_loss_region': stats_mh['region']}, step=b1_steps + mh_steps)


In [ ]:
# ── 13. Results ───────────────────────────────────────────────────────────────
print('=' * 50)
print(f'  VOC 2012 val mIoU  ({CFG["eval_images"]} images, MaskCLIP-style)')
print('=' * 50)
print(f'  B0  (frozen)         : {b0_miou:6.2f}%')
print(f'  B1  (λ_region=0)     : {b1_miou:6.2f}%')
print(f'  M_human (λ={CFG["lambda_region"]})     : {mh_miou:6.2f}%')
print()
print(f'  B1  − B0             : {b1_miou - b0_miou:+.2f}%')
print(f'  M_human − B1         : {mh_miou - b1_miou:+.2f}%   ← does L_region help?')
print(f'  M_human − B0         : {mh_miou - b0_miou:+.2f}%')
print('=' * 50)

print('\nTraining loss summary:')
print(f'  B1      global loss : {stats_b1["global"]:.3f}')
print(f'  M_human global loss : {stats_mh["global"]:.3f}')
print(f'  M_human region loss : {stats_mh["region"]:.3f}')

# Log final summary metrics and close the W&B run
wandb.summary.update({
    'b0_miou':          b0_miou,
    'b1_miou':          b1_miou,
    'mhuman_miou':      mh_miou,
    'delta_b1_vs_b0':   b1_miou - b0_miou,
    'delta_mhuman_vs_b1': mh_miou - b1_miou,
    'delta_mhuman_vs_b0': mh_miou - b0_miou,
})
wandb.finish()
print('W&B run finished.')
